# Mixed Precision Training Troubleshooting

## Common Issues and Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| Loss becomes NaN | Gradient overflow | Reduce init_scale, enable grad clipping |
| Training stalls | Gradient underflow | Increase init_scale |
| Accuracy drops | Precision loss | Keep sensitive ops in FP32 |

In [ ]:
import torch

def diagnose_mixed_precision(model, sample_input):
    """Diagnose mixed precision training issues."""
    
    # Check for problematic values
    with torch.cuda.amp.autocast():
        output = model(sample_input)
    
    # Check output range
    print(f"Output range: [{output.min():.2e}, {output.max():.2e}]")
    print(f"Output dtype: {output.dtype}")
    
    # Check for inf/nan
    if torch.isinf(output).any():
        print("WARNING: Output contains inf values!")
    if torch.isnan(output).any():
        print("WARNING: Output contains NaN values!")
    
    # Check gradient magnitudes
    loss = output.sum()
    loss.backward()
    
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.norm().item()
            if grad_norm < 1e-7:
                print(f"WARNING: {name} has very small gradient: {grad_norm:.2e}")
            elif grad_norm > 1e4:
                print(f"WARNING: {name} has very large gradient: {grad_norm:.2e}")

## Best Practices

1. **Start with BF16** if hardware supports it
2. **Monitor loss scale** - should stabilize after warmup
3. **Keep LayerNorm/Softmax in FP32** for stability
4. **Use gradient clipping** (max_norm=1.0)